# 5.02 从零实现 ACT

[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ROI-su7/vla-ladder/blob/main/notebooks/05_act/5.02_act.ipynb)


## 为什么要学

ACT 是 ALOHA 平台的"原配"算法：50 条示教就能学会穿线、开杯盖、装电池这类精细双臂任务。π0 等 VLA 继承了它最重要的思想：**动作分块**。本课：

1. 精读 ACT 的架构（对照原版和 LeRobot 的代码）；
2. 自己实现 ACT 的损失和时间集成；
3. 用实验搞清楚：**β 有多大作用？推理为什么取 z=0？时间集成什么时候有用、什么时候有害？**

---

## 1. ACT 架构

```text
                        ┌──────────── 训练时才有：CVAE 编码器（BERT 式 Transformer）────────────┐
 真实动作块 a_{1:k} ──┐ │  [CLS] [qpos] [a_1] ... [a_k]  ──Transformer──> CLS 输出 ──> μ, log σ² │
 关节状态 qpos ──────┼─┤                                                     z = μ + σ·ε        │
                      │ └───────────────────────────────────────────────────────────────────────┘
                      │                       （推理时 z = 0）
                      ▼                                  │
  策略（CVAE 解码器）：                                    ▼
   图像 ──ResNet18──> 特征图 15×20 ──> 300 个 token ─┐
   qpos ──Linear──> 1 个 token ──────────────────────┼─> Transformer 编码器 ──> memory
   z ──Linear──> 1 个 token ─────────────────────────┘                          │ 交叉注意力
   k 个可学习的 query（DETR 风格）──────────────────────> Transformer 解码器 <──┘
                                                               └─> Linear ──> 动作块 (k, 14)
损失 = L1(预测动作块, 真实动作块) + β · KL(q(z|·) || N(0, I))，原版 β = 10、k = 100
```

ALOHA 上的原版有 4 个相机、14 维动作（双臂各 6 个关节 + 1 个夹爪）。本课在二维障碍物环境上用**状态版** ACT（没有图像，只有位置），其余结构完全一致。完整实现见 [`vla_ladder/models/act.py`](../../vla_ladder/models/act.py)，核心部分如下（建议对照原版 `detr_vae.py` 读一遍）：

```py
class ACT(nn.Module):
    def encode_latent(self, state, actions, is_pad=None):
        """CVAE 编码器：[CLS, state, a_1..a_k] -> Transformer -> CLS 的输出 -> (mu, logvar)"""
        tokens = cat([cls_embed, cvae_state_proj(state), cvae_action_proj(actions)]) + 位置编码
        h = cvae_encoder(tokens, src_key_padding_mask=pad)     # 注意：nn.Transformer 的 padding 掩码 True = 忽略
        mu, logvar = latent_proj(h[:, 0]).chunk(2, -1)

    def decode(self, state, z):
        """策略：memory = 编码器([z, state] (+ 图像 token))；k 个 query 通过交叉注意力读 memory"""
        memory = encoder(cat([latent_out_proj(z), state_proj(state)]) + enc_pos)
        h = decoder(query_embed, memory)                       # DETR 风格：每个 query 负责输出动作块中的一步
        return action_head(h)                                  # (B, k, A)
```

> **DETR 风格的解码器**：解码器的输入不是"上一步的输出"（不是自回归的），而是 $k$ 个**可学习的 query 向量**，每个 query 负责一个时间步。所以 $k$ 个动作是**一次性并行**生成的。这与 π0 的动作专家很像：π0 的动作 token 也是一次性生成整个块。

---

## 2. 准备数据

In [ ]:
# 🔧 环境准备（每个 notebook 都先运行这一格）
# - 在 Colab 中：自动下载本项目代码
# - 在本地：自动把项目根目录加入 sys.path（也可以在根目录执行 `pip install -e .`）
import os, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/vla-ladder"):
        os.system("git clone -q --depth 1 https://github.com/ROI-su7/vla-ladder.git /content/vla-ladder")
    sys.path.insert(0, "/content/vla-ladder")
else:
    _p = os.path.abspath(".")
    while _p != os.path.dirname(_p) and not os.path.isdir(os.path.join(_p, "vla_ladder")):
        _p = os.path.dirname(_p)
    sys.path.insert(0, _p)
print("环境就绪 ✅")

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from vla_ladder.envs import ObstacleReachEnv, collect_demos
from vla_ladder.models.act import ACT
from vla_ladder.utils import set_seed, get_device, count_params

device = get_device()
K = 16                                   # 动作块长度（原版 ALOHA 用 100 步 = 2 秒）
env = ObstacleReachEnv(num_envs=100, seed=0)
demos = collect_demos(env, n_episodes=200, seed=0)

def make_chunks(demos, K):
    """返回 obs (N, 2)，动作块 (N, K, 2)，is_pad (N, K)：越界补齐的位置为 True（不计入损失）"""
    Os, As, Ps = [], [], []
    for d in demos:
        T = len(d["actions"])
        raw = np.arange(T)[:, None] + np.arange(K)[None]
        Os.append(d["obs"]); As.append(d["actions"][np.minimum(raw, T - 1)]); Ps.append(raw >= T)
    return [torch.tensor(np.concatenate(v)).to(device) for v in (Os, As, Ps)]

O, A, PAD = make_chunks(demos, K)
print("样本数:", len(O), " 动作块形状:", tuple(A.shape[1:]))

## 3. 训练

### 练习 1：ACT 的损失函数

利用模型的 `encode_latent` 和 `decode` 两个方法实现 ACT 的训练损失：

1. 编码器从 (状态, 真实动作块) 得到 $\mu,\log\sigma^2$；
2. 重参数化采样 $z$；
3. 解码出动作块，计算 **L1** 重建误差，**只在非 padding 的位置上平均**；
4. 加上 $\beta\cdot$KL。

In [ ]:
def act_loss(model, state, actions, is_pad, beta):
    """返回 (总损失, l1, kl)"""
    # TODO
    raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
def act_loss(model, state, actions, is_pad, beta):
    """返回 (总损失, l1, kl)"""
    mu, logvar = model.encode_latent(state, actions, is_pad)
    z = mu + torch.randn_like(mu) * (0.5 * logvar).exp()
    pred = model.decode(state, z)                                 # (B, K, A)
    l1 = (pred - actions).abs().mean(-1)                          # (B, K)
    valid = (~is_pad).float()
    l1 = (l1 * valid).sum() / valid.sum().clamp(min=1)
    kl = 0.5 * (mu.pow(2) + logvar.exp() - 1 - logvar).sum(-1).mean()
    return l1 + beta * kl, l1, kl

In [ ]:
# ✅ 自测：与参考实现 ACT.loss 一致（固定随机种子，保证采样到同一个 ε）
set_seed(0)
m = ACT(2, 2, K).to(device).eval()                         # eval：关闭 dropout，结果可复现
i = torch.arange(32, device=device)
torch.manual_seed(1); mine, _, _ = act_loss(m, O[i], A[i], PAD[i], beta=10.0)
torch.manual_seed(1); ref, _ = m.loss(O[i], A[i], PAD[i], kl_weight=10.0)
assert torch.allclose(mine, ref, atol=1e-5)
print("通过 ✅")

In [ ]:
def train_act(beta, steps=3000, lr=5e-4, seed=0):
    set_seed(seed)
    model = ACT(state_dim=2, action_dim=2, chunk_size=K).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    for step in range(steps):
        i = torch.randint(0, len(O), (256,), device=device)
        loss, l1, kl = act_loss(model, O[i], A[i], PAD[i], beta)
        opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 10.0); opt.step()
        if step % 1000 == 0 or step == steps - 1:
            print(f"  β={beta}  step {step:4d}  L1 {l1.item():.4f}  KL {kl.item():.4f}")
    return model.eval()

print("参数量:", count_params(ACT(2, 2, K)))
act_default = train_act(beta=10.0)     # ACT 论文的默认值
act_small = train_act(beta=0.01)

注意 β=10 时 **KL 迅速降到接近 0**：后验坍缩了，$z$ 没被使用（上一课的结论）。

---

## 4. 推理：分块执行 vs 时间集成

### 方式一：分块执行

每 $n$ 步查询一次策略，执行动作块的前 $n$ 步（$n\le k$）。

### 方式二：时间集成（Temporal Ensembling）

**每一步都**查询策略。对当前时刻 $t$，所有在过去 $k$ 步内做出的预测都包含一个"对 $t$ 时刻的动作建议"。把它们加权平均：

$$
a_t=\frac{\sum_i w_i\,\hat a_t^{(i)}}{\sum_i w_i},\qquad w_i=\exp(-m\cdot i)
$$

$i=0$ 是**最早**的那次预测（原版 ACT 的约定：旧预测权重更大，$m$ 越大越偏向旧预测、越平滑）。

### 练习 2：实现时间集成

In [ ]:
class TemporalEnsembler:
    def __init__(self, k, m=0.01):
        self.k, self.m = k, m
        self.buffer = []          # [(预测时刻 t0, 动作块 (N, k, A)), ...]

    def step(self, t, chunk):
        """t: 当前时刻；chunk: (N, k, A) 当前预测。返回 (N, A) 要执行的动作"""
        # TODO:
        # 1) 把 (t, chunk) 加入 buffer；去掉 t - t0 >= k 的旧预测（它们已经不覆盖时刻 t 了）
        # 2) preds = stack([c[:, t - t0] for (t0, c) in buffer])     (n, N, A)，按时间从早到晚
        # 3) w = exp(-m * arange(n))，归一化后加权求和
        raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
class TemporalEnsembler:
    def __init__(self, k, m=0.01):
        self.k, self.m = k, m
        self.buffer = []

    def step(self, t, chunk):
        """t: 当前时刻；chunk: (N, k, A) 当前预测。返回 (N, A) 要执行的动作"""
        self.buffer.append((t, chunk))
        self.buffer = [(t0, c) for (t0, c) in self.buffer if t - t0 < self.k]
        preds = torch.stack([c[:, t - t0] for (t0, c) in self.buffer])
        w = torch.exp(-self.m * torch.arange(len(self.buffer), dtype=preds.dtype, device=preds.device))
        return (w[:, None, None] * preds).sum(0) / w.sum()

In [ ]:
# ✅ 自测：手算一个例子
te = TemporalEnsembler(k=3, m=1.0)
c = lambda v: torch.full((1, 3, 1), float(v))
te.step(0, c(1.0)); te.step(1, c(2.0))
out = te.step(2, c(3.0))                         # 时刻 2：三次预测 1, 2, 3，权重 e^0, e^-1, e^-2
w = np.exp(-np.arange(3)); expect = (w * np.array([1, 2, 3])).sum() / w.sum()
assert np.isclose(out.item(), expect)
out = te.step(3, c(4.0))                         # 时刻 3：时刻 0 的预测已经过期
w = np.exp(-np.arange(3)); expect = (w * np.array([2, 3, 4])).sum() / w.sum()
assert np.isclose(out.item(), expect) and len(te.buffer) == 3
print("通过 ✅")

### 评估函数

In [ ]:
@torch.no_grad()
def evaluate(model, mode="chunk", exec_steps=8, sample_z=False, n=200, seed=7):
    """mode='chunk'：每 exec_steps 步查询一次；mode='te'：每步查询 + 时间集成"""
    env = ObstacleReachEnv(num_envs=n, seed=seed)
    obs = env.reset(); trajs = [obs.copy()]; t = 0
    te = TemporalEnsembler(K)
    while not env.done.all():
        chunk = model.predict(torch.tensor(obs, device=device), sample_z=sample_z)    # (n, K, 2)
        if mode == "te":
            obs, _, _, info = env.step(te.step(t, chunk).cpu().numpy()); t += 1; trajs.append(obs.copy())
        else:
            chunk = chunk.cpu().numpy()
            for k in range(exec_steps):
                obs, _, _, info = env.step(chunk[:, k]); t += 1; trajs.append(obs.copy())
    traj = np.stack(trajs, 1)
    succ = info["success"]
    went_left = (traj[:, :, 0].min(1) < -0.3)[succ].mean() if succ.any() else float("nan")   # 成功回合中从左边绕的比例（衡量多样性）
    return {"success": info["success"].mean(), "collide": info["collided"].mean(), "left": went_left}, traj

results = {}
for name, model in [("β=10", act_default), ("β=0.01", act_small)]:
    for sample_z in [False, True]:
        for mode in ["chunk", "te"]:
            r, _ = evaluate(model, mode, sample_z=sample_z)
            results[(name, sample_z, mode)] = r
            print(f"{name:7s} z={'采样' if sample_z else '0   '}  {mode:5s}  成功 {r['success']:.2f}  碰撞 {r['collide']:.2f}  左绕比例 {r['left']:.2f}")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
cases = [("β=10", act_default, False, "chunk"), ("β=10", act_default, False, "te"),
         ("β=0.01", act_small, True, "chunk"), ("β=0.01", act_small, True, "te")]
for ax, (name, model, sz, mode) in zip(axes, cases):
    r, traj = evaluate(model, mode, sample_z=sz, n=40)
    env.plot(list(traj), ax=ax, title=f"{name}, z={'sampled' if sz else '0'}, {mode}: succ={r['success']:.2f}")
plt.tight_layout(); plt.show()

### 读懂实验结果

（深度学习实验有随机性，你的数字可能和下面描述的略有不同，但规律是一样的。）

1. **β=10（论文默认值）：后验坍缩。** KL 很快降到 0，$z$ 被忽略，ACT 退化成一个"确定性的 L1 回归"。在起点处，它面对的是左右各一半的示教，输出要么接近"平均"，要么碰巧偏向某一边（L1 的最优解是中位数，对两点分布不唯一）。
   - **分块执行**（开环执行 8 步）：如果起点预测接近平均，整段动作直冲障碍物，成功率可能是 0。
   - **时间集成**（每步重新预测）：等价于**闭环控制**，和 3.02 课的单步 MSE 一样，第一步的微小偏差会被反馈放大，模型"顺势"选定一边，于是成功了。
   - 结论：**坍缩后的 ACT 之所以还能工作，很大程度上靠的是时间集成带来的每步闭环重规划**，而不是 CVAE。
2. **β=0.01：$z$ 真正编码了模式。** 取 $z=0$ 得到一个确定的模式（总是从同一边绕），分块执行也能成功；从先验采样 $z$，就能得到**两种**绕行方式（左绕比例明显大于 0），分块执行成功率依然很高。
3. **采样的 $z$ + 时间集成 = 冲突。** 每一步采样一个新 $z$，相邻两次预测可能一个向左、一个向右，时间集成把它们**平均**，又回到了"取平均"的老问题，碰撞率上升。**这是 ACT 推理时取 $z=0$ 的原因之一**：保证相邻预测来自同一个模式，时间集成只起平滑作用。

> **对真机的启示**：ACT 在 ALOHA 上大获成功，很大程度上依赖**动作分块 + 时间集成（闭环重规划）**，CVAE 的作用比论文宣称的小（社区里很多人观察到 KL 很快趋近 0）。如果你的任务真的有明显的多模态（比如线头可以从两侧对接），扩散/流匹配类的动作头（下一章、π0）更可靠，因为它们不依赖"每一步都重新推理"。

### 练习 3：时间集成的平滑效果

用 β=10 的模型，对比"每 8 步查询一次（chunk）"与"时间集成（te）"的**动作平滑度**：计算每条轨迹相邻两步动作差的平均范数（jerk 的简化版）。

In [ ]:
@torch.no_grad()
def action_smoothness(model, mode, n=100, seed=7):
    """返回所有回合中 ||a_t - a_{t-1}|| 的平均值（越小越平滑）"""
    env = ObstacleReachEnv(num_envs=n, seed=seed)
    obs = env.reset(); te = TemporalEnsembler(K); t = 0; acts = []
    # TODO: 仿照 evaluate 运行策略，把每一步实际执行的动作 (n, 2) 存进 acts；
    #       最后 acts = stack -> (T, n, 2)，计算相邻差的范数并求均值（只统计没结束的步可以忽略，直接全算即可）
    raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
@torch.no_grad()
def action_smoothness(model, mode, n=100, seed=7):
    """返回所有回合中 ||a_t - a_{t-1}|| 的平均值（越小越平滑）"""
    env = ObstacleReachEnv(num_envs=n, seed=seed)
    obs = env.reset(); te = TemporalEnsembler(K); t = 0; acts = []
    while not env.done.all():
        chunk = model.predict(torch.tensor(obs, device=device))
        if mode == "te":
            a = te.step(t, chunk).cpu().numpy()
            obs, _, _, _ = env.step(a); acts.append(a); t += 1
        else:
            chunk = chunk.cpu().numpy()
            for k in range(8):
                obs, _, _, _ = env.step(chunk[:, k]); acts.append(chunk[:, k]); t += 1
    acts = np.stack(acts)
    return np.linalg.norm(np.diff(acts, axis=0), axis=-1).mean()

In [ ]:
# ✅ 自测
s_chunk, s_te = action_smoothness(act_default, "chunk"), action_smoothness(act_default, "te")
print(f"动作变化量：分块执行 {s_chunk:.4f}，时间集成 {s_te:.4f}")
assert s_te <= s_chunk * 1.05
print("通过 ✅ 时间集成让动作更平滑（在块与块的衔接处尤其明显）")

---

## 小结

| 设计 | 作用 | 实验结论 |
|---|---|---|
| 动作分块 | 减少决策次数、动作连贯 | 让模型"整段地"选择一个模式 |
| CVAE（β） | 用 $z$ 表示模式 | β=10 时坍缩；β 小时 $z$ 才编码模式 |
| L1 损失 | 对离群值鲁棒 | 坍缩后可能挑一个模式、也可能接近平均：**不可靠** |
| z=0 推理 | 确定性输出 | 与时间集成配合，避免模式冲突 |
| 时间集成 | 平滑 + 每步闭环重规划 | 能挽救坍缩的 ACT；但每步都要推理，且与随机采样冲突 |

**思考题**：π0 推理一次约 70~100ms，动作块 50 步（50Hz 下 1 秒）。如果像 ACT 那样每一步都推理并做时间集成，需要多快的推理速度？π0 实际上怎么做？（提示：openpi 的 `action_horizon` 与客户端的 `replan_steps`；以及 FluxVLA / openpi 的 RTC 实时分块。）